In [74]:
import pandas as pd
import time
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [75]:
# load the enriched data

df = pd.read_csv('../data/final_labeled_data.csv')
df.head()

,review_text,rating,word_count,is_fake,is_duplicate,emojis,sentiment,scores,compound,label,review_len
0,Nice product for family of 4,4.0,6,0,1,NaN,Positive,"{'neg': 0.0, 'neu': 0.588, 'pos': 0.412, 'comp...",0.4215,2,6
1,Not Good product,1.0,3,0,1,NaN,Negative,"{'neg': 0.546, 'neu': 0.454, 'pos': 0.0, 'comp...",-0.3412,0,3
2,Works best with PPPoE type cable internet RJ45,5.0,8,0,1,NaN,Positive,"{'neg': 0.0, 'neu': 0.625, 'pos': 0.375, 'comp...",0.6369,2,8
3,nice,5.0,1,1,1,NaN,Positive,"{'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound...",0.4215,2,1
4,Sharp upper edges.,1.0,3,0,1,NaN,Negative,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0,3


In [76]:
# handle missing values in the 'review_text' column by filling them with an empty string

df['review_text'] = df['review_text'].fillna('')

In [77]:
# creates input features (X) and target variable (y) for model training

X = df['review_text']
y = df['sentiment']

In [78]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [79]:
# tf-idf vectorization
# Updated Vectorizer to include emojis and symbols
# The token_pattern now includes Unicode characters (emojis)

vectorizer = TfidfVectorizer(
    max_features=5000, 
    stop_words=None,
    token_pattern=r"[\w\U0001f600-\U0001f64f\U0001f300-\U0001f5ff]+" 
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [80]:
# models dictionary for training and comparison

models = {
    # CRITICAL: class_weight='balanced' fixes the positive bias
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    
    "Naive Bayes": MultinomialNB(),
    
    # Also add it to Linear SVM if you want better negative detection there too
    "Linear SVM": LinearSVC(class_weight='balanced',dual=False), 
    
    "Decision Tree": DecisionTreeClassifier(max_depth=20,class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=50, n_jobs=-1, class_weight='balanced')
}

In [81]:
comparison_results = []

for name, model in models.items():
    print(f"Training started for: {name}...")
    start_time = time.time()
    
    # Train the model
    model.fit(X_train_vec, y_train)
    
    # predict and accuracy calculation
    y_pred = model.predict(X_test_vec)
    accuracy = accuracy_score(y_test, y_pred)
    duration = time.time() - start_time
    
    comparison_results.append({'Model': name, 'Accuracy': accuracy, 'Time (sec)': duration})
    print(f"Total accuracy for {name}: {accuracy:.4f}")
    print(f"--- {name} Report ---")
    print(classification_report(y_test, y_pred))


Training started for: Logistic Regression...
Total accuracy for Logistic Regression: 0.8077
--- Logistic Regression Report ---
              precision    recall  f1-score   support

    Negative       0.63      0.85      0.73        67
     Neutral       0.18      0.18      0.18        11
    Positive       0.93      0.83      0.88       208

    accuracy                           0.81       286
   macro avg       0.58      0.62      0.59       286
weighted avg       0.83      0.81      0.81       286

Training started for: Naive Bayes...
Total accuracy for Naive Bayes: 0.7937
--- Naive Bayes Report ---
              precision    recall  f1-score   support

    Negative       0.78      0.37      0.51        67
     Neutral       0.00      0.00      0.00        11
    Positive       0.80      0.97      0.87       208

    accuracy                           0.79       286
   macro avg       0.53      0.45      0.46       286
weighted avg       0.76      0.79      0.75       286

Training

c:\Users\Vaishnavi Giri\Desktop\@GPP\Project\Product-Review-Sentiment-Analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Vaishnavi Giri\Desktop\@GPP\Project\Product-Review-Sentiment-Analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Vaishnavi Giri\Desktop\@GPP\Project\Product-Review-Sentiment-Analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

In [82]:
# To view the results in a table format
comparison_df = pd.DataFrame(comparison_results)
print("\n--- Final Model Comparison ---")
print(comparison_df.sort_values(by='Accuracy', ascending=False))


--- Final Model Comparison ---
                 Model  Accuracy  Time (sec)
4        Random Forest  0.839161    0.099082
2           Linear SVM  0.821678    0.000000
0  Logistic Regression  0.807692    0.020286
1          Naive Bayes  0.793706    0.000000
3        Decision Tree  0.559441    0.013011


In [83]:
# highest accuracy model selection
best_model_name = comparison_df.sort_values(by='Accuracy', ascending=False).iloc[0]['Model']
print(f"best model: {best_model_name}")

# best model object selection
best_model = models[best_model_name] 

# models saves in folder for later use in the app
joblib.dump(best_model, '../models/sentiment_model.pkl')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')

print("best model and vectorizer saved successfully!")

best model: Random Forest
best model and vectorizer saved successfully!
